# FastAPI Wrapper Around MLflow Model

## Topic Goal


The complete workflow is:

1. Train a customer churn model.
2. Evaluate the model.
3. Infer MLflow model signature.
4. Log the model with signature and input example in the **same MLflow run**.
5. Create `model_uri` from the same active run.
6. Register the model using the same `model_uri`.
7. Generate a production-style FastAPI application.
8. Generate a sample request JSON file.
9. Generate a Python API test client.
10. Provide commands to run and test the FastAPI service.

This is useful because real enterprise projects often do not expose MLflow directly.  
Instead, teams wrap the MLflow model inside a custom API layer such as FastAPI.


## 1. Import Libraries

We import MLflow, pandas, scikit-learn, and JSON utilities.

The notebook will generate:

```text
outputs/app.py
outputs/sample_request.json
outputs/test_client.py
```

These generated files can be used to run and test the FastAPI wrapper.


In [ ]:
# Import os to create folders and manage environment variables.
import os

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import json for saving sample request files.
import json

# Import MLflow for experiment tracking, model logging, and model registry.
import mlflow

# Import MLflow scikit-learn integration for logging sklearn pipelines.
import mlflow.sklearn

# Import infer_signature to capture model input/output schema.
from mlflow.models.signature import infer_signature

# Import pandas for reading and processing tabular data.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import train_test_split for splitting dataset into training and testing sets.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for separate preprocessing of categorical and numerical columns.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical variables.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical variables.
from sklearn.preprocessing import StandardScaler

# Import Pipeline for combining preprocessing and model together.
from sklearn.pipeline import Pipeline

# Import RandomForestClassifier for churn prediction.
from sklearn.ensemble import RandomForestClassifier

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Configure MLflow and Local Folders

This section prepares the local folder structure.

For classroom demos, this notebook uses local file-based MLflow tracking:

```text
file:./mlruns
```


In [ ]:
# Define experiment name.
EXPERIMENT_NAME = "advanced_10_fastapi_wrapper"

# Define MLflow run name.
RUN_NAME = "10_fastapi_wrapper_same_run_usecase"

# Remove inherited MLflow Project experiment ID if present.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited MLflow run ID if present.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create outputs folder for generated API files.
os.makedirs("outputs", exist_ok=True)

# Create artifacts folder for supporting files.
os.makedirs("artifacts", exist_ok=True)

# Configure MLflow local tracking URI.
mlflow.set_tracking_uri("file:./mlruns")

# Define dataset path.
DATA_PATH = "datasets/customer_churn_500.csv"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Define FastAPI host.
fastapi_host = "127.0.0.1"

# Define FastAPI port.
fastapi_port = 8000

# Define FastAPI base URL.
fastapi_url = f"http://{fastapi_host}:{fastapi_port}"

# Define prediction endpoint URL.
predict_url = f"{fastapi_url}/predict"

# Print configuration details.
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", RUN_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)
print("FastAPI URL:", fastapi_url)
print("Prediction URL:", predict_url)


## 3. Load and Prepare Dataset

We load the customer churn dataset.

The target column is:

```text
churn
```

The model predicts whether a customer is likely to churn.


In [ ]:
# Load customer churn dataset.
df = pd.read_csv(DATA_PATH)

# Display first five records.
display(df.head())

# Define target column.
target_column = "churn"

# Create feature matrix by dropping customer ID and target.
X = df.drop(columns=["customer_id", target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Print feature groups.
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Standardize numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split data into train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print dataset details.
print("Dataset shape:", df.shape)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 4. Train and Evaluate Model

We train a complete scikit-learn pipeline.

This is important because the FastAPI app should pass raw records to the MLflow model.  
The model itself should handle preprocessing internally.


In [ ]:
# Create Random Forest classifier.
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    random_state=42
)

# Create complete pipeline with preprocessing and model.
pipeline = Pipeline(steps=[
    # Apply preprocessing to raw input columns.
    ("preprocessor", preprocessor),

    # Train the Random Forest model.
    ("model", model),
])

# Train the pipeline.
pipeline.fit(X_train, y_train)

# Generate predictions on the test set.
predictions = pipeline.predict(X_test)

# Calculate accuracy.
accuracy = accuracy_score(y_test, predictions)

# Calculate precision.
precision = precision_score(y_test, predictions, zero_division=0)

# Calculate recall.
recall = recall_score(y_test, predictions, zero_division=0)

# Calculate F1-score.
f1 = f1_score(y_test, predictions, zero_division=0)

# Create classification report.
report = classification_report(y_test, predictions, zero_division=0)

# Print metrics.
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print(report)


## 5. Infer Model Signature

The signature captures the expected input and output format.

This is useful for API teams because they can understand what columns the model expects.


In [ ]:
# Create input example for signature.
input_example = X_test.head(5)

# Generate sample predictions for the input example.
sample_predictions = pipeline.predict(input_example)

# Infer MLflow model signature.
signature = infer_signature(input_example, sample_predictions)

# Display input example.
display(input_example)

# Print sample predictions.
print("Sample predictions:", sample_predictions)


## 6. Same-Run Model Logging, Signature, Model URI, and Registry

This section logs everything into the **same MLflow run**:

- parameters
- metrics
- classification report
- model artifact
- input example
- signature
- model URI

The model is then registered using that same `model_uri`.


In [ ]:
# Start the SAME MLflow run for metrics, model, signature, and model URI.
with mlflow.start_run(run_name=RUN_NAME):

    # Log dataset path.
    mlflow.log_param("dataset", DATA_PATH)

    # Log topic name.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Log FastAPI host.
    mlflow.log_param("fastapi_host", fastapi_host)

    # Log FastAPI port.
    mlflow.log_param("fastapi_port", fastapi_port)

    # Log FastAPI URL.
    mlflow.log_param("fastapi_url", fastapi_url)

    # Log accuracy.
    mlflow.log_metric("accuracy", accuracy)

    # Log precision.
    mlflow.log_metric("precision", precision)

    # Log recall.
    mlflow.log_metric("recall", recall)

    # Log F1-score.
    mlflow.log_metric("f1_score", f1)

    # Save classification report locally.
    with open("outputs/classification_report.txt", "w") as file:
        file.write(report)

    # Log classification report artifact.
    mlflow.log_artifact("outputs/classification_report.txt")

    # Log model WITH input example and signature.
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from the same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Register model using the same-run model URI.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

# Print same-run model URI.
print("Same-run model URI:", model_uri)

# Print registered model details.
print("Registered model name:", registered_model.name)
print("Registered model version:", registered_model.version)


## 7. Generate FastAPI Application

This cell generates:

```text
outputs/app.py
```

The FastAPI app includes:

- `/` home endpoint
- `/health` health-check endpoint
- `/model-info` endpoint
- `/predict` endpoint
- error handling
- model loading using `mlflow.pyfunc.load_model(model_uri)`


In [ ]:
# Create FastAPI application code.
fastapi_code = f'''# Import FastAPI framework.
from fastapi import FastAPI, HTTPException

# Import pydantic BaseModel for request validation.
from pydantic import BaseModel

# Import typing List and Dict for request schema.
from typing import List, Dict, Any

# Import pandas for converting incoming records into DataFrame.
import pandas as pd

# Import MLflow pyfunc for loading the logged model.
import mlflow.pyfunc

# Define MLflow model URI created from the same training run.
MODEL_URI = "{model_uri}"

# Create FastAPI application object.
app = FastAPI(
    title="Customer Churn MLflow API",
    description="FastAPI wrapper around an MLflow customer churn model",
    version="1.0.0"
)

# Load MLflow model once when application starts.
model = mlflow.pyfunc.load_model(MODEL_URI)


# Define request body schema.
class PredictionRequest(BaseModel):
    # records should be a list of dictionaries.
    records: List[Dict[str, Any]]


@app.get("/")
def home():
    # Return basic API message.
    return {{
        "message": "Customer Churn MLflow API is running",
        "docs_url": "/docs",
        "health_url": "/health"
    }}


@app.get("/health")
def health():
    # Return simple health status and model URI.
    return {{
        "status": "healthy",
        "model_uri": MODEL_URI
    }}


@app.get("/model-info")
def model_info():
    # Return model information for debugging/demo.
    return {{
        "model_uri": MODEL_URI,
        "model_interface": "mlflow.pyfunc",
        "expected_input": "List of customer records as dictionaries"
    }}


@app.post("/predict")
def predict(request: PredictionRequest):
    try:
        # Convert incoming list of dictionaries into a pandas DataFrame.
        input_df = pd.DataFrame(request.records)

        # Validate that input is not empty.
        if input_df.empty:
            raise HTTPException(status_code=400, detail="Input records cannot be empty.")

        # Generate predictions using the MLflow model.
        predictions = model.predict(input_df)

        # Return predictions as a list.
        return {{
            "prediction_count": len(predictions),
            "predictions": predictions.tolist()
        }}

    except HTTPException:
        # Re-raise HTTP exceptions directly.
        raise

    except Exception as error:
        # Return a clean API error message for unexpected failures.
        raise HTTPException(status_code=500, detail=str(error))
'''

# Save FastAPI app into outputs folder.
with open("outputs/app.py", "w") as file:
    file.write(fastapi_code)

# Print file creation message.
print("Generated FastAPI app: outputs/app.py")


## 8. Generate Sample Request JSON

The FastAPI `/predict` endpoint expects this format:

```json
{
  "records": [
    {
      "age": 35,
      "gender": "Male",
      ...
    }
  ]
}
```

This is different from MLflow serving `/invocations`, which expects `dataframe_split` or `dataframe_records`.


In [ ]:
# Select two sample records from test data.
sample_records = X_test.head(2).to_dict(orient="records")

# Create FastAPI request payload.
sample_request = {
    "records": sample_records
}

# Save sample request as JSON file.
with open("outputs/sample_request.json", "w") as file:
    json.dump(sample_request, file, indent=4)

# Print sample request.
print(json.dumps(sample_request, indent=4))


## 9. Generate Python Test Client

This cell generates:

```text
outputs/test_client.py
```

The test client calls the FastAPI `/predict` endpoint using `requests.post()`.


In [ ]:
# Create Python test client code.
test_client_code = f'''# Import json to load request payload.
import json

# Import requests to call FastAPI endpoint.
import requests

# Define API URL.
API_URL = "{predict_url}"

# Load sample request JSON file.
with open("sample_request.json", "r") as file:
    payload = json.load(file)

# Send POST request to FastAPI prediction endpoint.
response = requests.post(API_URL, json=payload, timeout=10)

# Print HTTP status code.
print("Status Code:", response.status_code)

# Try to print JSON response.
try:
    print("Response:", response.json())
except Exception:
    print("Raw Response:", response.text)
'''

# Save test client into outputs folder.
with open("outputs/test_client.py", "w") as file:
    file.write(test_client_code)

# Print file creation message.
print("Generated test client: outputs/test_client.py")


## 10. Commands to Run FastAPI App

Open a new terminal and go to the `outputs` folder:

```bash
cd outputs
```

Start the FastAPI server:

```bash
uvicorn app:app --reload --port 8000
```

Open Swagger UI:

```text
http://127.0.0.1:8000/docs
```

Run the Python test client:

```bash
python test_client.py
```


In [ ]:
# Create run command text.
run_commands = f'''
# Go to outputs folder
cd outputs

# Start FastAPI server
uvicorn app:app --reload --port {fastapi_port}

# Open Swagger UI in browser
http://127.0.0.1:{fastapi_port}/docs

# In another terminal from outputs folder, run test client
python test_client.py
'''

# Save run commands.
with open("outputs/run_fastapi_commands.txt", "w") as file:
    file.write(run_commands)

# Print commands.
print(run_commands)


## 11. Optional: Log Generated FastAPI Files as API Evidence

The model run already contains the model, signature, and metrics.

This optional run logs the generated API files as evidence:

- `app.py`
- `sample_request.json`
- `test_client.py`
- `run_fastapi_commands.txt`

This is separate from model training because API files are generated after model registration.


In [ ]:
# Start optional API evidence run.
with mlflow.start_run(run_name="fastapi_wrapper_evidence"):

    # Log model URI used by the FastAPI app.
    mlflow.log_param("model_uri", model_uri)

    # Log FastAPI URL.
    mlflow.log_param("fastapi_url", fastapi_url)

    # Log generated FastAPI app.
    mlflow.log_artifact("outputs/app.py")

    # Log sample request file.
    mlflow.log_artifact("outputs/sample_request.json")

    # Log test client file.
    mlflow.log_artifact("outputs/test_client.py")

    # Log run commands file.
    mlflow.log_artifact("outputs/run_fastapi_commands.txt")

# Print completion message.
print("FastAPI wrapper evidence logged.")


## 12. Trainer Notes



- MLflow model serving is useful, but enterprises often need custom API behavior.
- FastAPI lets us add request validation, custom endpoints, health checks, and business-specific response formats.
- The MLflow model is loaded using `mlflow.pyfunc.load_model(model_uri)`.
- The `/predict` endpoint receives JSON records and converts them to a pandas DataFrame.
- The logged model already contains preprocessing, so the API can pass raw input columns.
- Swagger UI at `/docs` is useful for testing the API interactively.
- The model run keeps metrics, model artifact, input example, signature, and registry lineage together.
- The optional API evidence run stores generated app files for reference.
